# Aula 2 — O pipeline dos anúncios: texto e imagem

Na aula 1 montamos a tubulação: um tópico do Pub/Sub, uma assinatura que grava no BigQuery e outra que
grava no Cloud Storage. Agora a gente coloca dado de verdade correndo por ela.

O fio condutor é **um anúncio de imóvel**. Ele entra no pipeline e se divide em dois caminhos:

- o **texto** (preço, área, endereço, descrição) vai pelo Pub/Sub e chega ao BigQuery, de onde saem as
  camadas Bronze / Silver / Gold que vocês já sabem construir;
- a **imagem** vai para o Cloud Storage, e de lá é o Gemini que tira dela a informação que o texto não
  traz.

No fim, os dois caminhos se reencontram no BigQuery.

> **Pré-requisito**: ter rodado o `gcp-pubsub-v2.ipynb` (aula 1) neste mesmo projeto. É de lá que vêm o
> tópico `aula-pdm-anuncios`, a tabela `aula_pdm.anuncios` e o bucket do Cloud Storage.

## 0. Configuração

In [ ]:
%pip install --quiet requests pydantic pillow google-cloud-pubsub google-cloud-storage google-cloud-bigquery google-genai

In [ ]:
from pathlib import Path
from urllib.request import urlopen

# Nos notebooks do BigQuery Studio só o .ipynb é enviado, então trazemos o módulo do crawler
# do repositório da aula. É o mesmo recurso que a aula 1 usa para baixar os dados.
modulo = Path("simple_crawler.py")
url = "https://raw.githubusercontent.com/robertogyn19/aula-pdm-pubsub/main/simple_crawler.py"

if modulo.exists():
    print(f"{modulo} já está aqui.")
else:
    modulo.write_bytes(urlopen(url).read())
    print(f"{modulo} baixado de {url}")

# A seção 3 depende de duas coisas que entraram no módulo junto com esta aula. Se o arquivo que
# chegou aqui for de uma versão anterior, é melhor descobrir agora do que no meio da aula.
import simple_crawler

for necessario in ("IMG_BASE_URL", "baixar_imagem"):
    if not hasattr(simple_crawler, necessario):
        raise ImportError(
            f"O {modulo} disponível não tem {necessario}. Apague o arquivo e rode esta célula "
            f"de novo para baixar a versão atual de {url}"
        )

print("simple_crawler pronto.")


In [ ]:
import google.auth

# O projeto vem da credencial do ambiente, como na aula 1.
_, project_id = google.auth.default()
print(project_id)

## 1. De onde vem o anúncio

Os anúncios da aula 1 chegaram prontos, num `.zip`. Eles saíram da API pública de listagem do
[Chaves na Mão](https://www.chavesnamao.com.br), e é essa coleta que a gente refaz agora — o pipeline
começa aqui, não no arquivo.

O código vive em `simple_crawler.py`, e é ele que este notebook importa. Assim existe uma única versão
da lógica, e o que você executa aqui é exatamente o que gerou os arquivos de `dados/anuncios`.

In [ ]:
from simple_crawler import ChavesNaMaoCrawler

crawler = ChavesNaMaoCrawler()

### 1.1. Uma página da API

O site expõe a listagem em JSON. A URL combina dois níveis de navegação — o tipo de negócio (`level1`)
e a localidade (`level2`) — mais o número da página.

In [ ]:
level1 = "imoveis-a-venda"
level2 = "go-goiania"

url = f"{crawler.base_url}{crawler.base_path}?level1={level1}&level2={level2}&pg=1"
url

In [ ]:
payload = crawler.coletar_pagina(url)
list(payload.keys())

### 1.2. A estrutura da resposta

A resposta tem duas partes: `items`, com os anúncios da página, e `metadata`, com as informações de
paginação. É o `metadata` que diz quantas páginas existem e qual é a próxima URL.

In [ ]:
print("anúncios nesta página:", len(payload["items"]))
print("total de páginas:", payload["metadata"]["totalPages"])
print("próxima página:", payload["metadata"]["links"]["nextApiParams"])

### 1.3. Do JSON para o modelo

O JSON da API é aninhado e irregular: preço, área e contagem de quartos aparecem em formatos diferentes
conforme o anúncio. O `extrair_dados` achata isso num modelo `Anuncio` do Pydantic, com tipos fixos — e
é esse formato achatado que a tabela `aula_pdm.anuncios` espera.

Nem todo item vira anúncio: os que não têm `id` são descartados.

In [ ]:
anuncios = crawler.extrair_dados(payload)
print(f"{len(anuncios)} anúncios extraídos de {len(payload['items'])} itens")

print(anuncios[0].model_dump_json(indent=2))

In [ ]:
# O modelo também tem comportamento, não só dados
anuncios[0].endereco()

### 1.4. Paginação

O `coletar_paginas` repete o processo acima seguindo a próxima URL de cada resposta, com uma pausa
entre as requisições.

Repare na diferença entre o que a API anuncia e o que ela entrega: o `metadata` desta consulta diz que
há mais de mil páginas, mas a listagem não devolve nada além da página 100. Quem trata esse limite é o
`_configura_ultima_pagina`, que corta em 100 quando ninguém pede um número menor.

O `ultima` abaixo está fixo em 2 de propósito — sem ele seriam cem requisições, coleta demais para o
tempo de uma aula.

In [ ]:
anuncios = crawler.coletar_paginas(level1, level2, primeira=1, ultima=2)
len(anuncios)

### 1.5. Como os arquivos da aula 1 foram gerados

O arquivo `dados/chavesnamao_level2.txt` tem a lista de estados e cidades usada na coleta. A função
`realizar_coleta_varias_paginas()`, no fim do `simple_crawler.py`, percorre essa lista e grava um
`.jsonl` por localidade em `dados/anuncios`, pulando os que já existem.

Foi assim que os 162 arquivos do `dados/anuncios.zip` foram produzidos, em setembro de 2025. Para
refazer a coleta, rode o script direto pelo terminal:

```bash
python simple_crawler.py
```

Repare no `__main__` do script: ele chama `realizar_coleta_varias_paginas()`, que percorre a lista de
localidades coletando `casas-a-venda` e gera os `anuncios_*.jsonl`. O `imoveis_go-goiania.jsonl` que a
aula 1 publica saiu da outra função, `realizar_coleta_goiania()`, que está comentada logo acima.


## 2. O caminho do texto: Pub/Sub → BigQuery

Este é o caminho que a aula 1 já deixou pronto. O tópico existe, a assinatura do BigQuery existe, e a
tabela existe. Não há nada para criar: basta publicar.

Repare no que **não** aparece aqui — nenhuma linha de `INSERT`, nenhuma conexão com o BigQuery. Quem
escreve na tabela é a assinatura, e ela já estava lá esperando.

In [ ]:
from google.cloud import pubsub_v1

topico_anuncios = f"projects/{project_id}/topics/aula-pdm-anuncios"

publisher = pubsub_v1.PublisherClient()
topico_anuncios

In [ ]:
# O model_dump_json() do Pydantic produz exatamente o formato que o esquema da tabela espera
futures = [
    publisher.publish(topico_anuncios, data=a.model_dump_json().encode("utf-8"))
    for a in anuncios
]

for fut in futures:
    fut.result(timeout=60)

print(f"{len(futures)} anúncios publicados em {topico_anuncios}")

### 2.1. Conferindo no BigQuery

A assinatura leva alguns instantes para escrever. Se o resultado vier vazio, espere um pouco e rode de
novo — e, se continuar vazio, o caminho de investigação é a DLQ da seção 5 da aula 1.

In [ ]:
from google.cloud import bigquery

cliente_bq = bigquery.Client(project=project_id)

# A tabela já tem os anúncios da aula 1, então uma contagem geral não diria nada sobre esta
# publicação. Perguntamos pelos ids que acabamos de publicar.
ids_publicados = ", ".join(str(a.id) for a in anuncios)

sql = f"""
SELECT COUNT(DISTINCT id) AS chegaram
FROM `{project_id}.aula_pdm.anuncios`
WHERE id IN ({ids_publicados})
"""

chegaram = list(cliente_bq.query(sql).result())[0].chegaram
print(f"{chegaram} dos {len(anuncios)} anúncios publicados já estão na tabela\n")

sql_cidades = f"""
SELECT cidade, COUNT(*) AS anuncios, ROUND(AVG(preco), 2) AS preco_medio
FROM `{project_id}.aula_pdm.anuncios`
GROUP BY cidade
ORDER BY anuncios DESC
LIMIT 5
"""

for linha in cliente_bq.query(sql_cidades).result():
    print(f"{linha.cidade:25s} {linha.anuncios:6d}  {linha.preco_medio}")


A partir daqui o texto já é território conhecido: essa tabela é a camada Bronze, e as
camadas Silver e Gold saem dela com o SQL que vocês já viram com o professor Sávio.

O que a tabela **não** tem é o que só a foto mostra. É o outro caminho.

## 3. O caminho da imagem: download → Cloud Storage

O texto foi em lote: publicamos todos os anúncios da coleta de uma vez, e a assinatura cuidou do
resto. Com as imagens vai ser diferente.

### 3.1. O anúncio que vamos seguir

Daqui em diante o notebook acompanha **um** anúncio, escolhido entre os que têm algumas fotos. Não é
limitação técnica: é para dar tempo de olhar cada foto ao lado do que o modelo responder sobre ela, na
seção 4. Escalar isso para o resto da coleta é o trabalho de casa.

In [ ]:
anuncio = next(a for a in anuncios if len(a.imagens) >= 3)

print(anuncio.id, "|", anuncio.titulo)
print(anuncio.endereco())
print(anuncio.preco_fmt, "|", anuncio.quartos, "quartos |", len(anuncio.imagens), "imagens")

### 3.2. Das URLs aos arquivos

A API não devolve as imagens, devolve o caminho de cada arquivo. A URL completa se monta juntando esse
caminho a uma base do site, e é isso que o `urls_imagens()` faz.

In [ ]:
for url_imagem in anuncio.urls_imagens():
    print(url_imagem)

### 3.3. Baixando as imagens

O `baixar_imagem` cuida de um detalhe chato: mesmo pedindo JPEG, o site às vezes responde WebP. Ele
detecta o formato pelo conteúdo e converte quando precisa, para todo arquivo daqui em diante ser JPEG.

In [ ]:
from simple_crawler import baixar_imagem

diretorio = Path(f"imagens/{anuncio.id}")

arquivos = [
    baixar_imagem(url_imagem, diretorio / f"{idx}.jpg")
    for idx, url_imagem in enumerate(anuncio.urls_imagens(), start=1)
]

print(f"{len(arquivos)} imagens em {diretorio}")

### 3.4. Enviando para o Cloud Storage

O bucket é o mesmo da seção 7 da aula 1. A célula abaixo o cria caso você não tenha feito aquela seção;
se ele já existir, não faz nada.

In [ ]:
bucket_name = f"{project_id}-aula-pdm"

!gcloud storage buckets describe gs://{bucket_name} --format="value(name)" 2>/dev/null || gcloud storage buckets create gs://{bucket_name} --location us-central1

In [ ]:
from google.cloud import storage

bucket = storage.Client(project=project_id).bucket(bucket_name)

for arquivo in arquivos:
    blob = bucket.blob(f"imagens/{anuncio.id}/{arquivo.name}")
    blob.upload_from_filename(arquivo)
    print(f"gs://{bucket_name}/{blob.name}")

As imagens agora estão no mesmo lugar em que a assinatura do GCS da aula 1 grava os
arquivos Avro, num prefixo diferente. Dois formatos, dois caminhos, um bucket.

## 4. O que só a imagem sabe

O texto do anúncio diz o preço, a área e o número de quartos. Ele não diz se o piso é porcelanato ou
taco — e é exatamente esse tipo de informação que muda uma avaliação de imóvel.

Vamos pedir isso ao Gemini.

### 4.1. Ligando a API e criando o cliente

O Gemini roda na Vertex AI, que precisa estar habilitada no projeto. É um comando só, e ele é idempotente.

In [ ]:
!gcloud services enable aiplatform.googleapis.com --project {project_id}

In [ ]:
from google import genai
from google.genai import types

# vertexai=True faz o SDK usar a mesma credencial que já autenticou o Pub/Sub, o Storage e o
# BigQuery neste notebook — não há chave de API para criar. O SDK descobriria o projeto e a
# região sozinho; estão explícitos aqui só para ficar visível de onde vêm.
cliente_gemini = genai.Client(vertexai=True, project=project_id, location="global")

MODELO = "gemini-2.5-flash"
MODELO

### 4.2. A pergunta

Duas decisões que valem mais que o código:

1. **Pergunte pouco.** Uma pergunta ampla ("descreva o imóvel") devolve texto bonito e inútil para uma
   tabela. Duas colunas concretas devolvem dado.
2. **Peça a resposta estruturada.** O `response_schema` faz o modelo responder no formato do
   `Caracteristica` abaixo, em vez de um parágrafo que alguém teria que decifrar depois.

In [ ]:
from pydantic import BaseModel


class Caracteristica(BaseModel):
    comodo: str
    piso: str


PERGUNTA = (
    "Esta é a foto de um anúncio de imóvel. "
    "Responda que cômodo aparece na foto e qual é o tipo de piso visível "
    "(por exemplo: cerâmica, porcelanato, laminado, taco, cimento queimado, madeira). "
    "Se não for possível identificar algum dos dois, responda NA nesse campo."
)

### 4.3. Uma imagem

Comece por uma só, para ver a foto e a resposta lado a lado.

In [ ]:
from IPython.display import Image as Exibir

primeira = arquivos[0]
Exibir(filename=primeira, width=420)

In [ ]:
resposta = cliente_gemini.models.generate_content(
    model=MODELO,
    contents=[
        types.Part.from_bytes(data=primeira.read_bytes(), mime_type="image/jpeg"),
        PERGUNTA,
    ],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Caracteristica,
    ),
)

# Com response_schema, o SDK já devolve o objeto pronto em .parsed
print(resposta.parsed)

Confira contra a foto acima. É aqui que se descobre se a pergunta está boa: se a
resposta estiver errada, o problema quase nunca é o modelo — é a pergunta ou a foto.

### 4.4. Todas as imagens do anúncio

O mesmo pedido, agora num laço sobre todas as fotos — a primeira inclusive, para a tabela da seção 5
sair completa. Uma chamada por imagem.

Repare que a primeira foto vai ser perguntada de novo. Se a resposta vier diferente da que apareceu na
seção 4.3, não é defeito: o modelo não é determinístico, e essa é uma das coisas que mudam quando um
passo do pipeline passa a ser um modelo em vez de uma função.

In [ ]:
extracoes = []

for arquivo in arquivos:
    resposta = cliente_gemini.models.generate_content(
        model=MODELO,
        contents=[
            types.Part.from_bytes(data=arquivo.read_bytes(), mime_type="image/jpeg"),
            PERGUNTA,
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Caracteristica,
        ),
    )
    caracteristica = resposta.parsed
    # O .parsed vem vazio quando o modelo não devolve JSON analisável (bloqueio de segurança,
    # limite de tokens). Numa aula, pular a foto é melhor que derrubar o laço no meio.
    if caracteristica is None:
        print(f"{arquivo.name}: o modelo não devolveu resposta analisável, pulando")
        continue

    extracoes.append({
        "anuncio_id": anuncio.id,
        "imagem_gcs": f"gs://{bucket_name}/imagens/{anuncio.id}/{arquivo.name}",
        "comodo": caracteristica.comodo,
        "piso": caracteristica.piso,
    })

for extracao in extracoes:
    print(f"{extracao['comodo']:20s} {extracao['piso']}")

## 5. Onde os dois caminhos se reencontram

O texto virou linha na `aula_pdm.anuncios`. A imagem virou arquivo no bucket e, agora, linha na
`aula_pdm.imagens_caracteristicas`. As duas tabelas se ligam pelo `anuncio_id`.

Essa segunda tabela é uma camada Bronze como a primeira: dado bruto, do jeito que a origem entregou,
sem limpeza. O que vem depois dela é o mesmo trabalho de sempre.

In [ ]:
tabela_imagens = f"{project_id}.aula_pdm.imagens_caracteristicas"

esquema = [
    bigquery.SchemaField("anuncio_id", "INTEGER"),
    bigquery.SchemaField("imagem_gcs", "STRING"),
    bigquery.SchemaField("comodo", "STRING"),
    bigquery.SchemaField("piso", "STRING"),
]

cliente_bq.create_table(bigquery.Table(tabela_imagens, schema=esquema), exists_ok=True)

erros = cliente_bq.insert_rows_json(tabela_imagens, extracoes)
print("erros:", erros or "nenhum")
print(f"{len(extracoes)} linhas gravadas em {tabela_imagens}")

### 5.1. A consulta que junta os dois caminhos

O `insert_rows_json` grava pela API de streaming do BigQuery, e a linha leva alguns instantes até ficar
visível para consulta. Se o resultado vier vazio, espere um pouco e rode a célula de novo — é o mesmo
comportamento da seção 2.1.

In [ ]:
# A tabela de anúncios é append-only: o mesmo id pode ter entrado na aula 1 e de novo agora.
# Sem o DISTINCT, cada par (cômodo, piso) apareceria uma vez por cópia do anúncio.
sql = f"""
SELECT DISTINCT
  a.id,
  a.titulo,
  a.preco,
  i.comodo,
  i.piso
FROM `{project_id}.aula_pdm.anuncios` AS a
JOIN `{project_id}.aula_pdm.imagens_caracteristicas` AS i
  ON i.anuncio_id = a.id
WHERE a.id = {anuncio.id}
"""

for linha in cliente_bq.query(sql).result():
    print(f"{linha.comodo:20s} {linha.piso:20s} {linha.titulo[:50]}")

Uma coluna dessa consulta não existia em lugar nenhum quando o anúncio foi coletado.
Ela foi produzida no meio do pipeline — e é isso que a aula queria mostrar.

## 6. Trabalho para casa

O que fizemos aqui foi um anúncio, duas colunas e uma pergunta curta. O trabalho é levar isso a sério.

1. **Escale a coleta.** Rode o crawler para mais de uma cidade e publique tudo no tópico. Quantas
   mensagens por segundo a assinatura do BigQuery aguenta? O que aparece na DLQ quando você força um
   anúncio com campo fora do esquema?

2. **Escale as imagens.** Uma chamada por imagem, em série, não vai terminar nunca para mil anúncios.
   Pense em lote, em paralelismo e no que fazer quando uma chamada falha no meio do caminho.

3. **Melhore a extração.** Duas colunas é pouco. Em `arquivo/ml/` está o material que o professor usou
   para avaliação de imóveis: um prompt com tabela de pontuação para estrutura, esquadrias, piso,
   forro, cobertura e benfeitorias, com nota para cada opção. Adapte-o para o formato estruturado que
   usamos na seção 4.

   Repare que a nossa pergunta deixou o modelo responder com as palavras que quisesse; a tabela tem uma
   lista fechada de opções, com nota para cada uma. Fazer o modelo responder dentro dessa lista é parte
   do trabalho.

4. **Suba as camadas.** Com Bronze de texto e Bronze de imagem no BigQuery, construa Silver e Gold.
   Qual pergunta de negócio a Gold responde que nenhuma das duas Bronze responde sozinha?